# Notebook for fetching one file from the bulk-files API to understand the structure

In [38]:
import httpx
import os
from pathlib import Path
from dotenv import load_dotenv
from urllib.parse import urljoin
import pandas as pd

In [39]:
load_dotenv()
api_key = os.environ.get("API_KEY")
base = os.environ.get("BASE")
bulk = os.environ.get("BULK").rstrip("/") + "/"

In [40]:
bulk

'https://api.epa.gov/easey/bulk-files/'

In [41]:
headers = {"x-api-key": api_key}
r = httpx.get(base, headers=headers, timeout=60)
files = r.json()

In [42]:
items = files["items"]
print(f"Total files in listing: {len(items)}")

Total files in listing: 235454


In [43]:
data_types = {item["metadata"].get("dataType") for item in items}
print(f"Data types: {data_types}")

Data types: {'Allowance', 'Facility', 'Emissions', 'Compliance', 'EDR', 'XML', 'Mercury and Air Toxics Emissions (MATS)'}


In [44]:
emissions = [it for it in items if it["metadata"].get("dataType") == "Emissions"]
print(f"Emissions files: {len(emissions)}")

Emissions files: 3209


In [45]:
for it in emissions[:5]:
    print(it["filename"], "->", it["metadata"])

emissions-hourly-2018-ri.csv -> {'description': 'Unit-level Hourly Part 75 emissions data for all RI facilities/units for 2018', 'year': '2018', 'stateCode': 'RI', 'dataType': 'Emissions', 'dataSubType': 'Hourly'}
emissions-daily-2018-ut.csv -> {'description': 'Unit-level Daily Part 75 emissions data for all UT facilities/units for 2018', 'year': '2018', 'stateCode': 'UT', 'dataType': 'Emissions', 'dataSubType': 'Daily'}
emissions-daily-2018-wv.csv -> {'description': 'Unit-level Daily Part 75 emissions data for all WV facilities/units for 2018', 'year': '2018', 'stateCode': 'WV', 'dataType': 'Emissions', 'dataSubType': 'Daily'}
emissions-daily-2018-sd.csv -> {'description': 'Unit-level Daily Part 75 emissions data for all SD facilities/units for 2018', 'year': '2018', 'stateCode': 'SD', 'dataType': 'Emissions', 'dataSubType': 'Daily'}
emissions-hourly-2018-ut.csv -> {'description': 'Unit-level Hourly Part 75 emissions data for all UT facilities/units for 2018', 'year': '2018', 'stateCo

In [46]:
NON_CONUS = {"AK", "HI", "PR", "VI", "GU", "MP", "AS"}
daily = [
    it
    for it in items
    if it["metadata"].get("dataType") == "Emissions"
    and it["metadata"].get("dataSubType") == "Daily"
    and it["metadata"].get("year") in ["2021", "2022"]
    and it["metadata"].get("stateCode") not in NON_CONUS
]
print(f"Daily emissions files for 2021-2022, CONUS: {len(daily)}")
print("Expected: ~50 states × 2 years = ~100 files")

Daily emissions files for 2021-2022, CONUS: 106
Expected: ~50 states × 2 years = ~100 files


In [47]:
# Sanity check — show a few
for f in daily[:5]:
    md = f["metadata"]
    print(
        f"  {md['stateCode']} {md['year']}: {f['filename']} ({f['megaBytes']:.1f} MB)"
    )

  AL 2021: emissions-daily-2021-al.csv (6.8 MB)
  IA 2021: emissions-daily-2021-ia.csv (3.4 MB)
  KY 2021: emissions-daily-2021-ky.csv (6.7 MB)
  MI 2021: emissions-daily-2021-mi.csv (8.3 MB)
  MO 2021: emissions-daily-2021-mo.csv (7.4 MB)


In [48]:
total_mb = sum(f["megaBytes"] for f in daily)
print(f"\nTotal download size: {total_mb:.0f} MB")


Total download size: 1130 MB


In [49]:
print(daily[0]["filename"], daily[0]["metadata"], daily[0]["s3Path"])

emissions-daily-2021-al.csv {'description': 'Unit-level Daily Part 75 emissions data for all AL facilities/units for 2021', 'year': '2021', 'stateCode': 'AL', 'dataType': 'Emissions', 'dataSubType': 'Daily'} emissions/daily/state/emissions-daily-2021-al.csv


In [50]:
# Let's try to download just one file to see how it looks and confirm we can read it as expected.
# We'll use the s3Path field to get the file from S3, which should be a public bucket.
# We can use httpx for this as well since it's just an HTTP GET request.
target_file = daily[0]
download_url = urljoin(bulk, target_file["s3Path"])
out_dir = Path("data/raw/epa")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / target_file["filename"]
download_url

'https://api.epa.gov/easey/bulk-files/emissions/daily/state/emissions-daily-2021-al.csv'

In [51]:
with httpx.stream("GET", download_url, headers=headers, timeout=120) as resp:
    resp.raise_for_status()
    with open(out_path, "wb") as f:
        for chunk in resp.iter_bytes():
            f.write(chunk)

In [52]:
print("Saved:", out_path, out_path.stat().st_size, "bytes")

Saved: data/raw/epa/emissions-daily-2021-al.csv 7181998 bytes


In [53]:
df = pd.read_csv(out_path)
print(df.shape)

(33218, 25)


/tmp/ipykernel_13374/748598362.py:1: DtypeWarning: Columns (0: Associated Stacks, 1: SO2 Controls, 2: Hg Controls) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out_path)


In [54]:
print(df.columns.tolist())

['State', 'Facility Name', 'Facility ID', 'Unit ID', 'Associated Stacks', 'Date', 'Operating Time Count', 'Sum of the Operating Time', 'Gross Load (MWh)', 'Steam Load (1000 lb)', 'SO2 Mass (short tons)', 'SO2 Rate (lbs/mmBtu)', 'CO2 Mass (short tons)', 'CO2 Rate (short tons/mmBtu)', 'NOx Mass (short tons)', 'NOx Rate (lbs/mmBtu)', 'Heat Input (mmBtu)', 'Primary Fuel Type', 'Secondary Fuel Type', 'Unit Type', 'SO2 Controls', 'NOx Controls', 'PM Controls', 'Hg Controls', 'Program Code']


In [55]:
df.tail()

,State,Facility Name,Facility ID,Unit ID,Associated Stacks,Date,Operating Time Count,Sum of the Operating Time,Gross Load (MWh),Steam Load (1000 lb),...,NOx Rate (lbs/mmBtu),Heat Input (mmBtu),Primary Fuel Type,Secondary Fuel Type,Unit Type,SO2 Controls,NOx Controls,PM Controls,Hg Controls,Program Code
33213,AL,"WestRock CP, LLC - Stevenson Mill",880101,PB3,NaN,2021-09-26,21,21.0,NaN,4200.0,...,0.099,5670.0,Pipeline Natural Gas,Diesel Oil,Dry bottom wall-fired boiler,NaN,Low NOx Burner Technology (Dry Bottom only),NaN,NaN,SIPNOX
33214,AL,"WestRock CP, LLC - Stevenson Mill",880101,PB3,NaN,2021-09-27,24,24.0,NaN,4800.0,...,0.099,6480.0,Pipeline Natural Gas,Diesel Oil,Dry bottom wall-fired boiler,NaN,Low NOx Burner Technology (Dry Bottom only),NaN,NaN,SIPNOX
33215,AL,"WestRock CP, LLC - Stevenson Mill",880101,PB3,NaN,2021-09-28,11,11.0,NaN,2200.0,...,0.099,2970.0,Pipeline Natural Gas,Diesel Oil,Dry bottom wall-fired boiler,NaN,Low NOx Burner Technology (Dry Bottom only),NaN,NaN,SIPNOX
33216,AL,"WestRock CP, LLC - Stevenson Mill",880101,PB3,NaN,2021-09-29,24,24.0,NaN,4800.0,...,0.099,6480.0,Pipeline Natural Gas,Diesel Oil,Dry bottom wall-fired boiler,NaN,Low NOx Burner Technology (Dry Bottom only),NaN,NaN,SIPNOX
33217,AL,"WestRock CP, LLC - Stevenson Mill",880101,PB3,NaN,2021-09-30,24,24.0,NaN,4800.0,...,0.099,6480.0,Pipeline Natural Gas,Diesel Oil,Dry bottom wall-fired boiler,NaN,Low NOx Burner Technology (Dry Bottom only),NaN,NaN,SIPNOX


The sample file downloaded includes both `Gross Load` and `CO2 Mass` which are important for the paper replication

Let's explore whether there are biomass plants which we can filter out

In [56]:
all_files = sorted(Path("data/raw/epa_daily").absolute().glob("*.csv"))

In [57]:
from co2sat.utils import data_dir

In [58]:
all_files = sorted(data_dir("raw", "epa_daily").glob("*.csv"))
print(f"Found {len(all_files)} files.")

Found 106 files.


In [59]:
primary_fuels = set()
secondary_fuels = set()
unit_types = set()

In [60]:
cols = ["Primary Fuel Type", "Secondary Fuel Type", "Unit Type"]
dtypes = {c: "string" for c in cols}  # force string dtype

In [61]:
for f in all_files:
    df = pd.read_csv(f, usecols=cols, dtype=dtypes)
    primary_fuels.update(df["Primary Fuel Type"].dropna().unique())
    secondary_fuels.update(df["Secondary Fuel Type"].dropna().unique())

In [62]:
print("All Primary Fuel Type values across dataset:")
for f in sorted(primary_fuels):
    print(f"  {f!r}")

All Primary Fuel Type values across dataset:
  'Coal'
  'Coal Refuse'
  'Coal, Pipeline Natural Gas'
  'Diesel Oil'
  'Natural Gas'
  'Other Gas'
  'Other Oil'
  'Petroleum Coke'
  'Pipeline Natural Gas'
  'Process Gas'
  'Residual Oil'
  'Wood'


In [63]:
print(f"\nAll Secondary Fuel Type values ({len(secondary_fuels)}):")
for v in sorted(secondary_fuels):
    print(f"  {v!r}")


All Secondary Fuel Type values (47):
  'Coal'
  'Coal Refuse, Wood'
  'Coal, Other Solid Fuel'
  'Coal, Other Solid Fuel, Petroleum Coke, Tire Derived Fuel, Wood'
  'Coal, Pipeline Natural Gas'
  'Coal, Tire Derived Fuel'
  'Diesel Oil'
  'Diesel Oil, Liquified Petroleum Gas'
  'Diesel Oil, Natural Gas'
  'Diesel Oil, Natural Gas, Other Gas'
  'Diesel Oil, Natural Gas, Pipeline Natural Gas'
  'Diesel Oil, Other Gas'
  'Diesel Oil, Other Oil'
  'Diesel Oil, Other Oil, Pipeline Natural Gas'
  'Diesel Oil, Other Solid Fuel'
  'Diesel Oil, Pipeline Natural Gas'
  'Diesel Oil, Residual Oil'
  'Diesel Oil, Wood'
  'Liquified Petroleum Gas, Pipeline Natural Gas'
  'Natural Gas'
  'Natural Gas, Pipeline Natural Gas'
  'Natural Gas, Tire Derived Fuel'
  'Natural Gas, Wood'
  'Other Gas'
  'Other Gas, Other Oil, Wood'
  'Other Gas, Pipeline Natural Gas'
  'Other Gas, Process Gas'
  'Other Gas, Process Gas, Waste Liquid'
  'Other Gas, Residual Oil'
  'Other Gas, Wood'
  'Other Oil'
  'Other Oil,

In [64]:
print(f"\nAll Unit Type values ({len(unit_types)}):")
for v in sorted(unit_types):
    print(f"  {v!r}")


All Unit Type values (0):


In [75]:
FOSSIL_PRIMARY_FUELS = {
    "Coal",
    "Coal Refuse",
    "Coal, Pipeline Natural Gas",
    "Diesel Oil",
    "Natural Gas",
    "Other Gas",
    "Other Oil",
    "Petroleum Coke",
    "Pipeline Natural Gas",
    "Process Gas",
    "Residual Oil",
}


# Excluded: "Wood" (biomass)
def project_root() -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError("Could not find project root")


ROOT = project_root()
PROCESSED_DIR = ROOT / "data" / "processed"

parquet_file = PROCESSED_DIR / "epa_daily.parquet"
print("Reading from:", parquet_file)
print("Exists:", parquet_file.exists())

df = pd.read_parquet(parquet_file)
df.info()

Reading from: /home/karim/co2-satellite-replication/data/processed/epa_daily.parquet
Exists: True
<class 'pandas.DataFrame'>
RangeIndex: 970308 entries, 0 to 970307
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   facility_id        970308 non-null  int64         
 1   state              970308 non-null  str           
 2   date               970308 non-null  datetime64[us]
 3   co2_metric_tons    970308 non-null  float64       
 4   gross_load_mwh     970308 non-null  float64       
 5   primary_fuel_type  970308 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(2)
memory usage: 61.3 MB


In [78]:
df_fossil = df[
    (df["co2_metric_tons"] > 0)
    & (df["gross_load_mwh"] > 0)
    & (df["primary_fuel_type"].isin(FOSSIL_PRIMARY_FUELS))
].copy()

PERIODS = [
    ("2021-04-01", "2021-05-20"),
    ("2021-09-01", "2021-10-01"),
    ("2022-04-01", "2022-04-30"),
    ("2022-09-01", "2022-09-29"),
]
PAPER_PLANTS = [583, 590, 513, 592]
PAPER_SAMPLES = [20306, 14464, 11213, 15243]

print(
    f"{'Period':30s} {'Plants':>8s} {'(paper)':>8s} {'Δ':>6s} {'Rows':>7s} {'(paper)':>8s}"
)
print("-" * 75)
for (start, end), pp, ps in zip(PERIODS, PAPER_PLANTS, PAPER_SAMPLES):
    sub = df_fossil[(df_fossil["date"] >= start) & (df_fossil["date"] < end)]
    n_p = sub["facility_id"].nunique()
    n_r = len(sub)
    print(
        f"  {start} to {end[:7]:8s}  {n_p:8d} {pp:8d} {n_p - pp:+5d} {n_r:7d} {ps:8d}"
    )

# Also see how many wood-only plants we excluded
wood_plants = df[df["primary_fuel_type"] == "Wood"]["facility_id"].nunique()
print(f"\nExcluded wood-primary plants: {wood_plants}")

Period                           Plants  (paper)      Δ    Rows  (paper)
---------------------------------------------------------------------------
  2021-04-01 to 2021-05       1039      583  +456   29564    20306
  2021-09-01 to 2021-10       1067      590  +477   21726    14464
  2022-04-01 to 2022-04        987      513  +474   17611    11213
  2022-09-01 to 2022-09       1047      592  +455   21439    15243

Excluded wood-primary plants: 20
